# PaperMind — Notebook 6: Multi-Modal (Figures)

Research papers are not just text. The most information-dense parts are often the **figures**: architecture diagrams, attention-pattern visualisations, performance plots, ablation tables. A text-only RAG pipeline silently throws all of that away.

This notebook closes the gap. We:

1. Acquire figure images (we'll try a direct URL first, then fall back to rendering the right pages from the paper PDF).
2. Send each image to a **multi-modal Gemini model** with text prompts.
3. **Persist the model's descriptions as text Documents** with metadata pointing back to the figure.
4. Re-index those description Documents alongside the paper's text in a single `VectorStoreIndex`.
5. Run a query that retrieves both figure-derived context AND paper-text context — the bridge that makes "what does Figure 1 show?" answerable from a text query engine.

**A note on packages:** The brief asked for `llama-index-multi-modal-llms-google`, but that package was retired with Google's old `google.generativeai` SDK. We use `llama-index-llms-google-genai` (already in `requirements.txt`), which exposes multi-modal calls via `ChatMessage` + `ImageBlock` — same SDK we use for text-only Gemini work. Same story for the model: `gemini-1.5-flash` was retired in late 2025; we use `gemini-2.5-flash-lite` (handles vision, generous free-tier TPM).

## 1. Setup — env, multi-modal Gemini, embeddings

`GoogleGenAI` is the same class we used for text-only Gemini. Multi-modal happens through the `ChatMessage` block API:

```python
ChatMessage(role="user", blocks=[TextBlock(text=...), ImageBlock(path=...)])
```

We keep BGE for embeddings — text retrieval is unchanged.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

# Gemini's SDK calls asyncio.run() internally, which fails in a running
# Jupyter event loop. nest_asyncio patches asyncio to allow nested loops.
# (We avoided this in notebook 4's Groq agent because Groq's httpx+sniffio
# stack doesn't tolerate the patch — but Gemini does.)
nest_asyncio.apply()

load_dotenv("../.env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in ../.env"

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# gemini-2.5-flash-lite handles both text and image inputs and has the
# friendliest free-tier TPM among current Gemini models.
llm = GoogleGenAI(model="gemini-2.5-flash-lite", api_key=GEMINI_API_KEY)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

print("Multi-modal Gemini + BGE embeddings configured")

Multi-modal Gemini + BGE embeddings configured


## 2. Acquire figure images

Two figures from *Attention Is All You Need* (Vaswani et al., 2017):
- **Figure 1** — the Transformer encoder–decoder architecture diagram.
- **Figure 2** — Scaled Dot-Product Attention + Multi-Head Attention.

We first try the direct arXiv HTML image URLs. If those 404 (arXiv's HTML view doesn't always host extracted figure PNGs), we **fall back to rendering the corresponding pages from the paper's PDF** using PyMuPDF. Either way, we end up with two PNGs in `../data/figures/`.

Page-rendering produces an image that includes the figure plus its caption and a bit of surrounding text — that's actually useful, since the caption gives the model linguistic context to anchor its description.

In [2]:
import requests
import fitz  # pymupdf

FIGURES_DIR = Path("../data/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ATTENTION_PDF = Path("../data/attention.pdf")
assert ATTENTION_PDF.exists(), "Run notebook 2 first to download attention.pdf"

# Direct URLs to attempt first; PDF page numbers (1-indexed) for fallback.
figure_specs = [
    {"name": "figure1", "url": "https://arxiv.org/html/1706.03762/extracted/1706.03762-Figure1-1.png", "pdf_page": 3, "caption": "Transformer architecture"},
    {"name": "figure2", "url": "https://arxiv.org/html/1706.03762/extracted/1706.03762-Figure2-1.png", "pdf_page": 4, "caption": "Scaled dot-product + multi-head attention"},
]

for spec in figure_specs:
    target = FIGURES_DIR / f"{spec['name']}.png"
    spec["path"] = target

    if target.exists():
        print(f"[{spec['name']}] already on disk ({target.stat().st_size // 1024} KB)")
        continue

    # 1. Try direct URL
    try:
        r = requests.get(spec["url"], timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        if r.ok and r.headers.get("Content-Type", "").startswith("image/"):
            target.write_bytes(r.content)
            print(f"[{spec['name']}] downloaded from arXiv ({len(r.content) // 1024} KB)")
            continue
        print(f"[{spec['name']}] arXiv URL returned {r.status_code}; falling back to PDF")
    except Exception as e:
        print(f"[{spec['name']}] arXiv URL failed ({e!r}); falling back to PDF")

    # 2. Fall back to rendering the PDF page
    pdf = fitz.open(str(ATTENTION_PDF))
    page = pdf.load_page(spec["pdf_page"] - 1)
    pix = page.get_pixmap(dpi=150)
    pix.save(str(target))
    pdf.close()
    print(f"[{spec['name']}] rendered from PDF page {spec['pdf_page']} ({target.stat().st_size // 1024} KB)")

[figure1] already on disk (230 KB)
[figure2] already on disk (262 KB)


## 3. Load figures with `SimpleDirectoryReader`

`SimpleDirectoryReader` scans a directory and auto-classifies files. For images it returns `ImageDocument` instances that carry the file path and basic metadata.

For multi-modal calls we mostly just need the path — but loading via `SimpleDirectoryReader` is the canonical pattern, so we use it here both to confirm the figures are picked up and so the downstream code can iterate over `ImageDocument` objects rather than raw paths.

In [3]:
from llama_index.core import SimpleDirectoryReader

image_documents = SimpleDirectoryReader(input_dir=str(FIGURES_DIR)).load_data()
print(f"Loaded {len(image_documents)} image document(s):")
for doc in image_documents:
    fp = doc.metadata.get("file_path") or doc.metadata.get("file_name", "?")
    print(f"  • {Path(fp).name}")

Loaded 2 image document(s):
  • figure1.png
  • figure2.png


## 4. Multi-modal queries — describe each figure

Three prompts, each crafted for a different downstream use:

1. **Alt-text** — a short, accessibility-style description (one or two sentences). Useful for citations, captions, screen readers.
2. **Architecture components & connections** — a structural breakdown that names parts of the diagram and how they relate. This is the kind of text we want sitting next to the figure in the index.
3. **Reproduction-ready notes** — what a researcher would need from the figure to reimplement it.

We send each (image, prompt) pair to Gemini using `ChatMessage` with a `TextBlock` followed by an `ImageBlock`.

In [4]:
from llama_index.core.llms import ChatMessage
from llama_index.core.base.llms.types import ImageBlock, TextBlock

PROMPTS = {
    "alt_text": "Describe this figure as alternative text for a research paper. Keep it to 1–2 sentences.",
    "components": "What architecture components are shown in this figure, and how do they connect? Use a short bulleted list.",
    "reproduction": "What would a researcher need to know to reproduce what's shown here? Be concrete about shapes, dimensions, and operations.",
}


def describe_figure(image_path: Path, prompt: str) -> str:
    msg = ChatMessage(role="user", blocks=[
        TextBlock(text=prompt),
        ImageBlock(path=str(image_path)),
    ])
    response = llm.chat([msg])
    return response.message.content or ""


# {figure_name: {prompt_key: description_text}}
figure_descriptions: dict[str, dict[str, str]] = {}

for spec in figure_specs:
    print("=" * 100)
    print(f"[{spec['name']}] {spec['caption']}  ({spec['path']})")
    figure_descriptions[spec["name"]] = {}
    for key, prompt in PROMPTS.items():
        print(f"\n  prompt: {key}")
        text = describe_figure(spec["path"], prompt)
        figure_descriptions[spec["name"]][key] = text
        print(f"  →\n{text}\n")

[figure1] Transformer architecture  (../data/figures/figure1.png)

  prompt: alt_text
  →
The figure illustrates the Transformer model architecture, depicting stacked encoder and decoder layers. Each layer consists of multi-head attention and feed-forward sub-layers, with residual connections and layer normalization.


  prompt: components
  →
Here are the architecture components shown in the figure and how they connect:

*   **Input Embeddings:** Takes the input sequence and converts it into a vector representation.
*   **Positional Encoding:** Adds positional information to the input embeddings.
*   **Encoder Stack (Nx):**
    *   **Input Embedding + Positional Encoding:** The initial input to the encoder stack.
    *   **Add & Norm:** Combines the output of the sub-layer with its input and normalizes it.
    *   **Multi-Head Attention:** Allows the model to attend to different parts of the input sequence.
    *   **Add & Norm:** Combines the output of the Multi-Head Attention sub-la

## 5. Persist figure descriptions as text Documents

Each (figure × prompt) response becomes a `Document`. Metadata records:

- `figure` — which figure the description is about (so we can filter or display in retrieval results)
- `prompt` — which prompt produced the description (alt_text / components / reproduction)
- `source_type="figure_description"` — distinguishes from raw paper text in the index
- `image_path` — pointer back to the original image

This is the **annotation memory** pattern: visual content is converted to text once, persisted, and from that point on can be retrieved by ordinary text similarity. We also save a JSON snapshot to disk so we don't have to re-run the LLM calls.

In [5]:
import json
from llama_index.core import Document

DESCRIPTIONS_FILE = FIGURES_DIR / "descriptions.json"
DESCRIPTIONS_FILE.write_text(json.dumps(figure_descriptions, indent=2))
print(f"Saved descriptions JSON → {DESCRIPTIONS_FILE}")

figure_docs: list[Document] = []
for spec in figure_specs:
    for prompt_key, text in figure_descriptions[spec["name"]].items():
        figure_docs.append(Document(
            text=f"[{spec['caption']}, {prompt_key}]\n{text}",
            metadata={
                "figure": spec["name"],
                "prompt": prompt_key,
                "source_type": "figure_description",
                "source_paper": "Attention Is All You Need",
                "image_path": str(spec["path"]),
            },
        ))

print(f"\nBuilt {len(figure_docs)} figure-description Documents")
for d in figure_docs:
    print(f"  • figure={d.metadata['figure']}  prompt={d.metadata['prompt']}  ({len(d.text)} chars)")

Saved descriptions JSON → ../data/figures/descriptions.json

Built 6 figure-description Documents
  • figure=figure1  prompt=alt_text  (265 chars)
  • figure=figure1  prompt=components  (3271 chars)
  • figure=figure1  prompt=reproduction  (9759 chars)
  • figure=figure2  prompt=alt_text  (401 chars)
  • figure=figure2  prompt=components  (936 chars)
  • figure=figure2  prompt=reproduction  (5395 chars)


## 6. Re-index — figures + paper text in one `VectorStoreIndex`

We pull the paper text directly via PyMuPDF (same loader as previous notebooks) and combine it with the figure-description Documents. Both kinds of node land in the same vector store, sharing the same embedding space — which is what makes cross-modal retrieval work.

The key piece is the `source_type` metadata. After retrieval, we can tell the user *where* a chunk came from — "this came from a description of Figure 1" vs "this is a paragraph from page 3".

In [6]:
from llama_index.core import VectorStoreIndex

# Paper text — one Document per page, same as notebook 2.
pdf = fitz.open(str(ATTENTION_PDF))
paper_docs = [
    Document(
        text=page.get_text(),
        metadata={
            "page": i + 1,
            "source_type": "paper_text",
            "source_paper": "Attention Is All You Need",
        },
    )
    for i, page in enumerate(pdf)
]
pdf.close()

combined_docs = figure_docs + paper_docs
combined_index = VectorStoreIndex.from_documents(combined_docs)
print(f"Combined index built: {len(figure_docs)} figure-description docs + {len(paper_docs)} paper-text pages")

Combined index built: 6 figure-description docs + 15 paper-text pages


## 7. Bridge query — retrieve figure context alongside text context

A question like *"What does Figure 1 of the Transformer paper show, and how does the multi-head attention block fit into that architecture?"* genuinely needs both:
- the **figure description** for the diagram-level answer,
- the **paper text** for how multi-head attention is described in the Methods section.

We retrieve top-5 chunks and print each one with its `source_type` so you can see the index pulling from both kinds of source.

In [7]:
query_engine = combined_index.as_query_engine(similarity_top_k=5)

bridge_query = (
    "What does Figure 1 of the Transformer paper show, and how does the multi-head "
    "attention block described in the text fit into that architecture?"
)

response = query_engine.query(bridge_query)

print("=" * 100)
print(f"Q: {bridge_query}\n")
print(f"A: {response}\n")
print("--- Source nodes (mixed figure-description + paper-text) ---")
for i, n in enumerate(response.source_nodes, 1):
    src_type = n.node.metadata.get("source_type", "?")
    figure = n.node.metadata.get("figure")
    page = n.node.metadata.get("page")
    label = f"{src_type}"
    if figure:
        label += f" / {figure} ({n.node.metadata.get('prompt')})"
    if page:
        label += f" / page {page}"
    snippet = n.node.get_content()[:240].replace("\n", " ").strip()
    score = f"{n.score:.3f}" if n.score is not None else "n/a"
    print(f"\n[{i}] score={score}  {label}")
    print(f"    {snippet}...")

Q: What does Figure 1 of the Transformer paper show, and how does the multi-head attention block described in the text fit into that architecture?

A: Figure 1 illustrates the Transformer model's architecture, featuring stacked encoder and decoder layers. Each layer incorporates multi-head attention and feed-forward sub-layers, enhanced with residual connections and layer normalization.

The multi-head attention block is a key component within both the encoder and decoder stacks. In the encoder, it's a multi-head self-attention mechanism where the model attends to different parts of the input sequence. In the decoder, there are two types of multi-head attention: a masked multi-head self-attention that prevents attending to future positions, and a standard multi-head attention that allows the decoder to attend to the output of the encoder stack. These attention mechanisms are followed by residual connections and layer normalization.

--- Source nodes (mixed figure-description + paper-te

## Why bridging visual + text retrieval matters

**Plain text RAG silently discards a paper's most concentrated information.** A figure caption is rarely enough to answer questions about the figure — "what's in this diagram?" is unanswerable from the caption alone. With the figure-annotation pattern in this notebook:

- The visual content is converted to text **once**, by a multi-modal model, with metadata pointing back to the source image.
- From that point on, every text retriever in the system can see it. The router from notebook 2, the sub-question engine from notebook 3, the agent from notebooks 4–5 — all of them will retrieve figure-derived context the same way they retrieve any other chunk.
- Provenance is preserved: `source_type="figure_description"` plus `image_path` lets the UI render the actual figure next to the answer when a user clicks through.

**Tuning levers:**
1. **The prompts you use to describe figures.** "Describe this figure" is a weak prompt. Targeted prompts ("list the components and connections", "what dimensions are shown?") produce text that matches the queries users will eventually ask.
2. **Granularity.** One Document per (figure × prompt) keeps retrieval focused. Concatenating all descriptions into one giant Document hurts top-k precision.
3. **Refresh policy.** If you upgrade the multi-modal model, re-run the descriptions and reindex — the image path metadata is the join key.

**What this doesn't do:**
- Live image retrieval. The user types a query, we retrieve the *text* description, and the UI shows the cached image. We are NOT embedding image pixels alongside text. That's a separate technique (CLIP, SigLIP) — useful when users want to query *by image*, but unnecessary for a paper-Q&A system where queries are textual.
- Tables and structured data. Pages with dense tables benefit from a separate extraction step (e.g. `pymupdf` table extraction or a vision LLM prompted for structured output) before the same annotation-memory pattern kicks in.